# LeetCode 2798 — Number of Employees Who Met the Target bằng Python

Link đề: https://leetcode.com/problems/number-of-employees-who-met-the-target/ · Độ khó: **Easy** · Chủ đề: Array, Counting

Skeleton LeetCode:

```python
class Solution:
    def numberOfEmployeesWhoMetTarget(self, hours: List[int], target: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm **số nhân viên** làm việc **ít nhất `target` giờ** (tức `hours[i] >= target`).
2. Nắm khái niệm **đếm theo điều kiện**: vòng `for` với biến đếm, biểu thức sinh với `sum`, danh sách lọc với `len`.
3. Phân biệt rõ **`>=` (ít nhất)** và **`>` (nhiều hơn)**, và biết các trường hợp biên: `target = 0` (mọi người đạt), `target` lớn hơn mọi giờ làm (không ai đạt).
4. Đi từ **vòng lặp đếm** → **`sum` của các giá trị boolean** → **lọc rồi `len`** → **sắp xếp + tìm kiếm nhị phân (`bisect_left`)** → **mảng đếm theo giá trị (bucket) + tổng hậu tố**.
5. Hiểu **tìm kiếm nhị phân** ở mức nhập môn (tự cài `lower_bound`) và vì sao `bisect_left` đúng cho điều kiện `>=` còn `bisect_right` cho `>`.
6. Tránh các lỗi: dùng `>` thay `>=`, trả về danh sách thay vì số, dùng nhầm `bisect_right`, và so sánh cả danh sách với số.

## 2. Đề bài & ví dụ

Một công ty có `n` nhân viên, đánh số từ `0` tới `n − 1`. Mỗi nhân viên `i` đã làm việc `hours[i]` giờ trong công ty.

Công ty yêu cầu mỗi nhân viên làm **ít nhất `target` giờ**.

Cho mảng số nguyên không âm `hours` độ dài `n` và một số nguyên không âm `target`. Trả về **số nhân viên** đã làm việc **ít nhất `target` giờ**.

| Ví dụ | hours, target | Output | Giải thích |
|---|---|---|---|
| 1 | `[0,1,2,3,4]`, `2` | `3` | Công ty cần mỗi nhân viên làm ít nhất 2 giờ. Nhân viên 0 làm 0 giờ, nhân viên 1 làm 1 giờ: chưa đạt. Nhân viên 2, 3, 4 làm 2, 3, 4 giờ: đạt. Có 3 nhân viên đạt. |
| 2 | `[5,1,4,2,2]`, `6` | `0` | Công ty cần mỗi nhân viên làm ít nhất 6 giờ. Không ai làm đủ 6 giờ nên đáp án `0`. |

Constraints:

- `1 <= n == hours.length <= 50`
- `0 <= hours[i], target <= 10^5`

In [1]:
hours, target = [0, 1, 2, 3, 4], 2
for i, h in enumerate(hours):
    print(f"nhân viên {i}: {h} giờ ->", "đạt" if h >= target else "chưa đạt")

nhân viên 0: 0 giờ -> chưa đạt
nhân viên 1: 1 giờ -> chưa đạt
nhân viên 2: 2 giờ -> đạt
nhân viên 3: 3 giờ -> đạt
nhân viên 4: 4 giờ -> đạt


## 3. Kiến thức nền

### 3.1. Phép so sánh `>=`, `>` và hệ quả "ít nhất"

- `a >= b`: `a` **lớn hơn hoặc bằng** `b`.
- `a > b`: `a` **lớn hơn hẳn** `b`.

"Ít nhất `target` giờ" nghĩa là **bằng `target` cũng đạt**, nên dùng `>=`. Dùng `>` sẽ loại những người làm đúng `target` giờ.

In [2]:
print(2 >= 2, 2 > 2)     # True False
print(3 >= 2, 3 > 2)     # True True
print(1 >= 2, 1 > 2)     # False False

True False
True True
False False


### 3.2. Biến đếm trong vòng lặp `for`

Muốn đếm có bao nhiêu phần tử thoả điều kiện, ta khởi tạo `count = 0`, duyệt từng phần tử, mỗi khi điều kiện đúng thì `count += 1`.

In [3]:
count = 0
for h in [0, 1, 2, 3, 4]:
    if h >= 2:
        count += 1
        print("đạt:", h, "-> count =", count)
print("kết quả:", count)

đạt: 2 -> count = 1
đạt: 3 -> count = 2
đạt: 4 -> count = 3
kết quả: 3


### 3.3. `True` là `1`, `False` là `0`: cộng boolean

Trong Python, `True == 1` và `False == 0`; vì vậy `sum` của một dãy boolean chính là **số lượng `True`**. Biểu thức sinh `(h >= target for h in hours)` tạo dãy boolean mà không cần danh sách trung gian.

In [4]:
print(True + True, True + False)           # 2 1
print(sum([True, False, True, True]))      # 3
hours, target = [0, 1, 2, 3, 4], 2
print(sum(h >= target for h in hours))     # 3

2 1
3
3


### 3.4. Lọc bằng danh sách rồi `len`

`[h for h in hours if h >= target]` tạo danh sách các giá trị đạt; `len(...)` là số lượng. Cách này tốn thêm bộ nhớ cho danh sách trung gian nhưng dễ đọc và hữu ích khi cần cả danh sách.

In [5]:
hours, target = [0, 1, 2, 3, 4], 2
ok = [h for h in hours if h >= target]
print(ok, len(ok))        # [2, 3, 4] 3

[2, 3, 4] 3


### 3.5. `sorted` và vì sao sắp xếp giúp đếm

Sau khi sắp xếp **tăng dần**, những phần tử `>= target` nằm **liền nhau ở cuối** mảng. Nếu biết vị trí `p` của phần tử đầu tiên `>= target` thì đáp án là `n − p`.

In [6]:
hours = [5, 1, 4, 2, 2]
s = sorted(hours)
print(s)                  # [1, 2, 2, 4, 5]
# với target = 4: phần tử đầu tiên >= 4 là s[3] = 4, nên có 5 - 3 = 2 người đạt
print(len(s) - 3)         # 2

[1, 2, 2, 4, 5]
2


### 3.6. Tìm kiếm nhị phân và `bisect_left`

**Tìm kiếm nhị phân** tìm vị trí trong mảng đã sắp xếp bằng cách luôn **loại một nửa** phạm vi: so sánh phần tử giữa với đích; nếu nó còn nhỏ hơn `target` thì loại nửa trái, ngược lại loại nửa phải.

`bisect.bisect_left(a, x)` trả về **vị trí nhỏ nhất** `p` sao cho `a[p] >= x` (hoặc `len(a)` nếu không có). Còn `bisect_right(a, x)` trả về vị trí nhỏ nhất sao cho `a[p] > x`. Hai hàm khác nhau chỉ khi `x` xuất hiện trong `a`.

In [7]:
from bisect import bisect_left, bisect_right
a = [1, 2, 2, 4, 5]
print(bisect_left(a, 2), bisect_right(a, 2))    # 1 3   (phần tử 2 chiếm vị trí 1..2)
print(bisect_left(a, 3), bisect_right(a, 3))    # 3 3   (3 không có)
print(bisect_left(a, 6), bisect_right(a, 6))    # 5 5   (lớn hơn mọi phần tử -> len)
print(bisect_left(a, 0), bisect_right(a, 0))    # 0 0   (nhỏ hơn mọi phần tử -> 0)

1 3
3 3
5 5
0 0


In [8]:
# Tự cài lower_bound (giống bisect_left) để hiểu nó chạy thế nào
def lower_bound(a, x):
    lo, hi = 0, len(a)            # tìm trong nửa mở [lo, hi)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1          # a[mid] còn nhỏ hơn x: đáp án nằm bên phải mid
        else:
            hi = mid              # a[mid] >= x: có thể là đáp án, thu hẹp về bên trái
    return lo

a = [1, 2, 2, 4, 5]
for x in [0, 1, 2, 3, 4, 5, 6]:
    assert lower_bound(a, x) == bisect_left(a, x)
    print(f"x={x}: lower_bound = {lower_bound(a, x)}")

x=0: lower_bound = 0
x=1: lower_bound = 0
x=2: lower_bound = 1
x=3: lower_bound = 3
x=4: lower_bound = 3
x=5: lower_bound = 4
x=6: lower_bound = 5


### 3.7. Mảng đếm (bucket) và tổng hậu tố

Khi giá trị nằm trong khoảng nhỏ (ở đây `0..10^5`), ta có thể dùng một mảng `cnt` mà `cnt[v]` là **số nhân viên làm đúng `v` giờ**. Số người làm `>= target` giờ là `cnt[target] + cnt[target + 1] + ...` (tổng một **hậu tố**). Cắt lát `cnt[target:]` lấy đúng phần đó; nếu `target` vượt cỡ mảng thì lát cắt rỗng và `sum` là `0`.

In [9]:
hours = [5, 1, 4, 2, 2]
cnt = [0] * (max(hours) + 1)
for h in hours:
    cnt[h] += 1
print(cnt)               # [0, 1, 2, 0, 1, 1]
print(sum(cnt[4:]))      # 2   (4 và 5)
print(sum(cnt[6:]))      # 0   (target = 6 > max)

[0, 1, 2, 0, 1, 1]
2
0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đây là bài **đếm có điều kiện** đơn giản nhất: "Có bao nhiêu phần tử thoả `hours[i] >= target`?".

Tính tay ví dụ 1: `hours = [0, 1, 2, 3, 4]`, `target = 2`.

| nhân viên | giờ | `giờ >= 2`? | đếm |
|---|---|---|---|
| 0 | 0 | không | 0 |
| 1 | 1 | không | 0 |
| 2 | 2 | **có** (bằng 2) | 1 |
| 3 | 3 | **có** | 2 |
| 4 | 4 | **có** | 3 |

Kết quả `3`. Điều đáng chú ý: nhân viên số 2 làm **đúng 2 giờ** vẫn được tính, vì "ít nhất 2 giờ" bao gồm cả `2`.

Hai trường hợp biên: nếu `target = 0` thì **mọi** nhân viên đạt (giờ làm không âm); nếu `target` lớn hơn giờ lớn nhất thì **không ai** đạt.

In [10]:
hours = [5, 1, 4, 2, 2]
for target in [0, 1, 2, 4, 5, 6]:
    print(f"target={target}: số người đạt = {sum(h >= target for h in hours)}")

target=0: số người đạt = 5
target=1: số người đạt = 5
target=2: số người đạt = 4
target=4: số người đạt = 2
target=5: số người đạt = 1
target=6: số người đạt = 0


## 5. Cách 1 — Brute Force / baseline: vòng `for` với biến đếm

**Ý tưởng:** duyệt từng nhân viên; nếu `hours[i] >= target` thì tăng `count`. Cuối cùng trả về `count`.

In [11]:
from typing import List

class SolutionLoop:
    def numberOfEmployeesWhoMetTarget(self, hours: List[int], target: int) -> int:
        count = 0
        for h in hours:
            if h >= target:
                count += 1
        return count

In [12]:
sol = SolutionLoop()
print(sol.numberOfEmployeesWhoMetTarget([0, 1, 2, 3, 4], 2))     # 3
print(sol.numberOfEmployeesWhoMetTarget([5, 1, 4, 2, 2], 6))     # 0

3
0


**Độ phức tạp:** mỗi phần tử một phép so sánh ⇒ Time `O(n)`, Space `O(1)`.

**Nhận xét:** đây đã là cách tối ưu về mặt thời gian trong trường hợp tổng quát (phải nhìn mỗi phần tử ít nhất một lần). Các cách sau chỉ khác về **cách viết** hoặc **chuẩn bị dữ liệu để trả lời nhiều truy vấn `target`**.

## 6. Ý tưởng thuật toán / bước đột phá

Vì cách 1 đã `O(n)`, "bước đột phá" ở đây là **góc nhìn** giúp ta viết gọn hơn hoặc mở rộng cho nhiều truy vấn:

**Đột phá 1 — đếm boolean.** `True` là `1` nên `sum(h >= target for h in hours)` đếm trực tiếp, không cần biến phụ.

**Đột phá 2 — lọc rồi `len`.** Tách thành hai bước: lọc các phần tử đạt, rồi lấy độ dài. Có lợi khi ta cũng cần chính các phần tử đó.

**Đột phá 3 — sắp xếp một lần, trả lời nhiều truy vấn.** Nếu có **nhiều** giá trị `target` cần hỏi cho cùng `hours`, hãy sắp xếp `hours` một lần rồi mỗi truy vấn chỉ tốn `O(log n)` bằng `bisect_left`: số người đạt là `n − bisect_left(sorted_hours, target)`.

**Đột phá 4 — mảng đếm theo giá trị.** Khi miền giá trị nhỏ, dựng `cnt[v]` rồi tổng hậu tố cho đáp án mỗi truy vấn trong `O(1)` (sau khi tiền xử lý).

In [13]:
hours = [5, 1, 4, 2, 2]
s = sorted(hours)
from bisect import bisect_left
for target in [0, 2, 4, 6]:
    p = bisect_left(s, target)
    print(f"target={target}: vị trí đầu tiên >= target là {p}, số người đạt = {len(s)} - {p} = {len(s) - p}")

target=0: vị trí đầu tiên >= target là 0, số người đạt = 5 - 0 = 5
target=2: vị trí đầu tiên >= target là 1, số người đạt = 5 - 1 = 4
target=4: vị trí đầu tiên >= target là 3, số người đạt = 5 - 3 = 2
target=6: vị trí đầu tiên >= target là 5, số người đạt = 5 - 5 = 0


## 7. Các cách tiếp theo

### Cách 2 — `sum` của các giá trị boolean (code nộp)

**Từng đoạn:** `(h >= target for h in hours)` là biểu thức sinh các giá trị `True/False`; `sum` cộng chúng (mỗi `True` là `1`).

In [14]:
class SolutionSum:
    def numberOfEmployeesWhoMetTarget(self, hours: List[int], target: int) -> int:
        return sum(h >= target for h in hours)

In [15]:
sol = SolutionSum()
print(sol.numberOfEmployeesWhoMetTarget([0, 1, 2, 3, 4], 2))     # 3
print(sol.numberOfEmployeesWhoMetTarget([5, 1, 4, 2, 2], 6))     # 0

3
0


**Độ phức tạp:** Time `O(n)`, Space `O(1)` (biểu thức sinh không tạo danh sách).

### Cách 3 — Lọc bằng danh sách rồi `len`

**Từng đoạn:** `[h for h in hours if h >= target]` giữ các giờ đạt; `len(...)` đếm.

In [16]:
class SolutionFilter:
    def numberOfEmployeesWhoMetTarget(self, hours: List[int], target: int) -> int:
        met = [h for h in hours if h >= target]
        return len(met)

In [17]:
sol = SolutionFilter()
print(sol.numberOfEmployeesWhoMetTarget([0, 1, 2, 3, 4], 2))     # 3
print(sol.numberOfEmployeesWhoMetTarget([5, 1, 4, 2, 2], 6))     # 0

3
0


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (danh sách trung gian có thể dài tới `n`). Chọn cách này khi bạn muốn **xem** danh sách giờ đạt hoặc dùng lại.

### Cách 4 — Sắp xếp + tìm kiếm nhị phân `bisect_left`

**Từng đoạn:**

- `s = sorted(hours)`.
- `p = bisect_left(s, target)`: vị trí đầu tiên có `s[p] >= target`.
- Các phần tử từ `p` đến cuối đều `>= target` (vì đã sắp xếp), nên đáp án `len(s) − p`.

Dùng `bisect_left` (không phải `bisect_right`): với `x = target` có trong mảng, `bisect_left` trả vị trí **đầu tiên** của nhóm các giá trị bằng `target`, nên nhóm đó được tính vào đáp án; `bisect_right` sẽ bỏ qua nhóm này.

In [18]:
from bisect import bisect_left

class SolutionBisect:
    def numberOfEmployeesWhoMetTarget(self, hours: List[int], target: int) -> int:
        s = sorted(hours)
        return len(s) - bisect_left(s, target)

In [19]:
sol = SolutionBisect()
print(sol.numberOfEmployeesWhoMetTarget([0, 1, 2, 3, 4], 2))     # 3
print(sol.numberOfEmployeesWhoMetTarget([5, 1, 4, 2, 2], 6))     # 0
print(sol.numberOfEmployeesWhoMetTarget([2, 2, 2], 2))            # 3 (cả nhóm bằng target đều đạt)

3
0
3


**Độ phức tạp:** sắp xếp `O(n log n)`, tìm nhị phân `O(log n)` ⇒ Time `O(n log n)`, Space `O(n)`. Cho **một** truy vấn thì chậm hơn cách 1; **rất lợi** khi có nhiều truy vấn trên cùng `hours` (mỗi truy vấn thêm chỉ `O(log n)`).

### Cách 5 — Mảng đếm theo giá trị + tổng hậu tố

**Từng đoạn:**

- `cnt = [0] * (max(hours) + 1)`; với mỗi `h`: `cnt[h] += 1`.
- Đáp án là `sum(cnt[target:])`: tổng số người có giờ `>= target`. Nếu `target > max(hours)` thì `cnt[target:]` rỗng ⇒ `0`.

In [20]:
class SolutionBucket:
    def numberOfEmployeesWhoMetTarget(self, hours: List[int], target: int) -> int:
        cnt = [0] * (max(hours) + 1)
        for h in hours:
            cnt[h] += 1
        return sum(cnt[target:])

In [21]:
sol = SolutionBucket()
print(sol.numberOfEmployeesWhoMetTarget([0, 1, 2, 3, 4], 2))     # 3
print(sol.numberOfEmployeesWhoMetTarget([5, 1, 4, 2, 2], 6))     # 0
print(sol.numberOfEmployeesWhoMetTarget([0, 0], 0))               # 2

3
0
2


**Độ phức tạp:** Time `O(n + V)` với `V = max(hours)` (có thể lên `10^5`), Space `O(V)`. Với bài này nó **không** tốt hơn cách 1 (`V` lớn hơn `n` nhiều), nhưng có tiền xử lý một lần rồi mỗi truy vấn `O(1)` nếu dựng sẵn tổng hậu tố. Nó thể hiện kỹ thuật **đếm theo giá trị (counting sort)**.

In [22]:
# Trả lời nhiều truy vấn trong O(1): dựng tổng hậu tố một lần
hours = [5, 1, 4, 2, 2]
V = max(hours)
cnt = [0] * (V + 2)                   # thêm 1 ô để suffix[V + 1] = 0
for h in hours:
    cnt[h] += 1
suffix = [0] * (V + 2)
for v in range(V, -1, -1):
    suffix[v] = suffix[v + 1] + cnt[v]
print(suffix)                          # suffix[v] = số người làm >= v giờ
for target in [0, 2, 4, 5]:
    print(target, suffix[target])

[5, 5, 4, 2, 2, 1, 0]
0 5
2 4
4 2
5 1


## 8. TRACE từng bước

Trace Cách 1 trên `hours = [5, 1, 4, 2, 2]`, `target = 2`:

| nhân viên `i` | `hours[i]` | `>= 2`? | `count` sau |
|---|---|---|---|
| 0 | 5 | có | 1 |
| 1 | 1 | không | 1 |
| 2 | 4 | có | 2 |
| 3 | 2 | **có** (bằng) | 3 |
| 4 | 2 | **có** (bằng) | 4 |

Trace Cách 4 trên cùng dữ liệu: `s = [1, 2, 2, 4, 5]`; `bisect_left(s, 2) = 1` (phần tử đầu tiên `>= 2` là `s[1]`); đáp án `5 − 1 = 4` ✓.

In [23]:
def trace_loop(hours, target):
    count = 0
    for i, h in enumerate(hours):
        ok = h >= target
        if ok:
            count += 1
        print(f"i={i}, hours[i]={h}, {h} >= {target}? {ok}, count={count}")
    print("kết quả:", count)
    return count

trace_loop([5, 1, 4, 2, 2], 2)

i=0, hours[i]=5, 5 >= 2? True, count=1
i=1, hours[i]=1, 1 >= 2? False, count=1
i=2, hours[i]=4, 4 >= 2? True, count=2
i=3, hours[i]=2, 2 >= 2? True, count=3
i=4, hours[i]=2, 2 >= 2? True, count=4
kết quả: 4


4

In [24]:
def trace_bisect(hours, target):
    s = sorted(hours)
    print("sorted:", s)
    lo, hi = 0, len(s)
    while lo < hi:
        mid = (lo + hi) // 2
        if s[mid] < target:
            print(f"lo={lo}, hi={hi}, mid={mid}: s[mid]={s[mid]} < {target} -> lo = {mid + 1}")
            lo = mid + 1
        else:
            print(f"lo={lo}, hi={hi}, mid={mid}: s[mid]={s[mid]} >= {target} -> hi = {mid}")
            hi = mid
    print(f"vị trí đầu tiên >= {target}: {lo}; đáp án = {len(s)} - {lo} = {len(s) - lo}")
    return len(s) - lo

trace_bisect([5, 1, 4, 2, 2], 2)
trace_bisect([5, 1, 4, 2, 2], 6)

sorted: [1, 2, 2, 4, 5]
lo=0, hi=5, mid=2: s[mid]=2 >= 2 -> hi = 2
lo=0, hi=2, mid=1: s[mid]=2 >= 2 -> hi = 1
lo=0, hi=1, mid=0: s[mid]=1 < 2 -> lo = 1
vị trí đầu tiên >= 2: 1; đáp án = 5 - 1 = 4
sorted: [1, 2, 2, 4, 5]
lo=0, hi=5, mid=2: s[mid]=2 < 6 -> lo = 3
lo=3, hi=5, mid=4: s[mid]=5 < 6 -> lo = 5
vị trí đầu tiên >= 6: 5; đáp án = 5 - 5 = 0


0

## 9. Code nộp LeetCode

Dùng Cách 2 (`sum` của boolean): ngắn gọn, `O(n)` thời gian, `O(1)` bộ nhớ.

In [25]:
from typing import List

class Solution:
    def numberOfEmployeesWhoMetTarget(self, hours: List[int], target: int) -> int:
        # mỗi nhân viên làm >= target giờ đóng góp 1 (True == 1), ngược lại 0
        return sum(h >= target for h in hours)

In [26]:
print(Solution().numberOfEmployeesWhoMetTarget([0, 1, 2, 3, 4], 2))     # 3
print(Solution().numberOfEmployeesWhoMetTarget([5, 1, 4, 2, 2], 6))     # 0

3
0


## 10. Vì sao thuật toán đúng

**Cách 1 / 2 / 3.** Đáp án là `|{i : hours[i] >= target}|`. Cách 1 duyệt mọi `i` đúng một lần và cộng một cho mỗi `i` thoả điều kiện; cách 2 cộng `True = 1` cho mỗi `i` thoả điều kiện (và `0` cho `i` còn lại); cách 3 lọc đúng các `i` thoả rồi đếm độ dài. Cả ba đếm đúng tập chỉ số thoả mãn, mỗi chỉ số một lần.

**Cách 4 (sắp xếp + `bisect_left`).** Gọi `s` là `hours` đã sắp xếp tăng dần. Sắp xếp là một hoán vị nên **số phần tử `>= target` không đổi**. Trong `s`, tính chất "`s[j] >= target`" **đơn điệu**: nếu đúng tại `j` thì đúng với mọi `j' > j` (vì `s[j'] >= s[j]`). Do đó các `j` thoả tạo thành một đoạn cuối `[p, n − 1]`, với `p` là vị trí nhỏ nhất thoả `s[p] >= target`. `bisect_left(s, target)` trả đúng `p` theo định nghĩa. Số phần tử thoả là `n − p`. Nếu không có phần tử nào `>= target` thì `p = n` và đáp án `0` ✓; nếu mọi phần tử `>= target` thì `p = 0` và đáp án `n` ✓.

**Vì sao `bisect_left`, không `bisect_right`.** `bisect_right(s, target)` trả vị trí nhỏ nhất `q` thoả `s[q] > target`, tức **bỏ qua** các phần tử bằng `target`; `n − q` là số phần tử **lớn hơn hẳn** `target`: đáp án cho điều kiện `>`, không phải `>=`.

**Cách 5 (đếm theo giá trị).** `cnt[v]` là số nhân viên đúng `v` giờ; mỗi nhân viên đóng góp vào đúng một ô. Số nhân viên có giờ `>= target` là `Σ_{v >= target} cnt[v] = sum(cnt[target:])`. Nếu `target > max(hours)`, lát cắt rỗng cho `0` ✓.

Cách 1 là bản dịch thẳng của đề; test mục 12 kiểm chứng các cách còn lại khớp với nó.

## 11. Phân tích Time / Space Complexity

`n = len(hours)`, `V = max(hours)` (tối đa `10^5`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vòng `for` | `O(n)` | `O(1)` | cơ bản nhất |
| 2. `sum` boolean | `O(n)` | `O(1)` | **code nộp** |
| 3. Lọc rồi `len` | `O(n)` | `O(n)` | cần danh sách trung gian |
| 4. Sắp xếp + `bisect_left` | `O(n log n)` | `O(n)` | lợi khi nhiều truy vấn (`O(log n)` mỗi truy vấn) |
| 5. Mảng đếm | `O(n + V)` | `O(V)` | sau tiền xử lý, mỗi truy vấn `O(1)` với tổng hậu tố |

Với `n <= 50` mọi cách đều tức thì. Với một truy vấn duy nhất, cách 2 là tốt nhất; đáng nhớ cách 4 / 5 cho dạng bài **nhiều truy vấn**.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: hai ví dụ của đề; `target = 0` (mọi người đạt, kể cả người `0` giờ); `target` bằng đúng giờ của vài người (kiểm tra `>=`); `target` lớn hơn tất cả (`0`); một nhân viên; mọi giờ bằng nhau; giá trị lớn `10^5`.

In [27]:
sols = [SolutionLoop(), SolutionSum(), SolutionFilter(), SolutionBisect(), SolutionBucket(), Solution()]
cases = [
    ("ví dụ 1", [0, 1, 2, 3, 4], 2, 3),
    ("ví dụ 2", [5, 1, 4, 2, 2], 6, 0),
    ("target = 0", [0, 0, 5], 0, 3),
    ("target = 0 và hours = 0", [0], 0, 1),
    ("một người đạt", [10], 10, 1),
    ("một người chưa đạt", [9], 10, 0),
    ("bằng đúng target", [2, 2, 2], 2, 3),
    ("mọi giờ bằng nhau, thiếu 1", [4, 4, 4], 5, 0),
    ("target = max", [1, 7, 3], 7, 1),
    ("target = max + 1", [1, 7, 3], 8, 0),
    ("giá trị lớn", [100000, 99999, 100000], 100000, 2),
    ("target lớn nhất", [100000, 100000], 100000, 2),
    ("target = 10^5, không ai đạt", [99999, 0], 100000, 0),
    ("hours toàn 0, target 1", [0, 0, 0], 1, 0),
]
for name, hours, target, exp in cases:
    for s in sols:
        got = s.numberOfEmployeesWhoMetTarget(hours[:], target)
        assert got == exp, (name, type(s).__name__, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> 3
PASS ví dụ 2 -> 0
PASS target = 0 -> 3
PASS target = 0 và hours = 0 -> 1
PASS một người đạt -> 1
PASS một người chưa đạt -> 0
PASS bằng đúng target -> 3
PASS mọi giờ bằng nhau, thiếu 1 -> 0
PASS target = max -> 1
PASS target = max + 1 -> 0
PASS giá trị lớn -> 2
PASS target lớn nhất -> 2
PASS target = 10^5, không ai đạt -> 0
PASS hours toàn 0, target 1 -> 0


### 12.2. Phản ví dụ: `>` thay cho `>=`

Với `[2, 2, 2]`, `target = 2`, đáp án đúng là `3`; một lời giải dùng `>` trả `0`. Nếu dùng `bisect_right` thay `bisect_left` cũng lệch tương tự. Ta kiểm tra điều này để thấy bộ test của ta có bắt được các lỗi đó (mục 13 trình bày chi tiết).

In [28]:
from bisect import bisect_right
def dung_gt(hours, target):
    return sum(h > target for h in hours)

def dung_bisect_right(hours, target):
    s = sorted(hours)
    return len(s) - bisect_right(s, target)

assert Solution().numberOfEmployeesWhoMetTarget([2, 2, 2], 2) == 3
assert dung_gt([2, 2, 2], 2) == 0
assert dung_bisect_right([2, 2, 2], 2) == 0
print("test bắt được lỗi > và bisect_right")

test bắt được lỗi > và bisect_right


### 12.3. Đối chiếu ngẫu nhiên (hợp lệ theo đề)

Sinh `n` trong `1..50`, `hours[i]` trong `0..10^5`, `target` trong `0..10^5`. Vì giá trị ngẫu nhiên hiếm khi trùng, ta thêm kiểu sinh với miền nhỏ (`0..8`) để có nhiều giá trị bằng `target`, và kiểu sinh có cấu trúc: chọn `target` bằng đúng một giá trị trong `hours`. Ta kiểm các tính chất: `0 <= đáp án <= n`; không tăng khi `target` tăng.

In [29]:
import random
random.seed(2798)

def gen_wide():
    n = random.randint(1, 50)
    return [random.randint(0, 100000) for _ in range(n)], random.randint(0, 100000)

def gen_narrow():
    n = random.randint(1, 50)
    return [random.randint(0, 8) for _ in range(n)], random.randint(0, 9)

def gen_tie():
    n = random.randint(1, 50)
    hours = [random.randint(0, 100000) for _ in range(n)]
    return hours, random.choice(hours)

for _ in range(500):
    for gen in (gen_wide, gen_narrow, gen_tie):
        hours, target = gen()
        assert 1 <= len(hours) <= 50 and all(0 <= h <= 100000 for h in hours) and 0 <= target <= 100000
        exp = SolutionLoop().numberOfEmployeesWhoMetTarget(hours[:], target)
        for s in sols:
            got = s.numberOfEmployeesWhoMetTarget(hours[:], target)
            assert got == exp, (type(s).__name__, hours, target, got, exp)
        assert 0 <= exp <= len(hours)
        if target + 1 <= 100000:
            assert Solution().numberOfEmployeesWhoMetTarget(hours, target + 1) <= exp     # không tăng
        assert Solution().numberOfEmployeesWhoMetTarget(hours, 0) == len(hours)
        assert Solution().numberOfEmployeesWhoMetTarget(hours, max(hours) + 1) == 0
        snap = hours[:]
        Solution().numberOfEmployeesWhoMetTarget(hours, target)
        assert hours == snap
print("1500 ca ngẫu nhiên: 6 cách khớp nhau, các tính chất đều đúng")

1500 ca ngẫu nhiên: 6 cách khớp nhau, các tính chất đều đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `>` thay cho `>=`

In [30]:
def sai_lon_hon(hours: List[int], target: int) -> int:
    return sum(h > target for h in hours)       # sai: bỏ sót người làm đúng target giờ

print("sai :", sai_lon_hon([0, 1, 2, 3, 4], 2))                          # 2
print("đúng:", Solution().numberOfEmployeesWhoMetTarget([0, 1, 2, 3, 4], 2))  # 3

sai : 2
đúng: 3


**Vì sao sai:** "làm việc **ít nhất** `target` giờ" bao gồm cả người làm **đúng** `target` giờ. Phải dùng `>=`.

### Lỗi 2: Trả về danh sách thay vì số lượng

In [31]:
def sai_tra_danh_sach(hours: List[int], target: int):
    return [h for h in hours if h >= target]     # sai: đề hỏi SỐ nhân viên

print("sai :", sai_tra_danh_sach([0, 1, 2, 3, 4], 2))                    # [2, 3, 4]
print("đúng:", Solution().numberOfEmployeesWhoMetTarget([0, 1, 2, 3, 4], 2))  # 3

sai : [2, 3, 4]
đúng: 3


**Vì sao sai:** kết quả phải là một **số nguyên** (số nhân viên). Bọc `len(...)` hoặc dùng `sum`.

### Lỗi 3: Dùng `bisect_right` thay `bisect_left`

In [32]:
from bisect import bisect_left, bisect_right
s = sorted([5, 1, 4, 2, 2])
print("bisect_left :", len(s) - bisect_left(s, 2))     # 4 (đúng)
print("bisect_right:", len(s) - bisect_right(s, 2))    # 2 (sai: bỏ hai người làm đúng 2 giờ)

bisect_left : 4
bisect_right: 2


**Vì sao sai:** `bisect_right(s, 2)` đi **qua** mọi giá trị bằng `2`, nên các người làm đúng `2` giờ không được đếm. Với điều kiện `>=` phải dùng `bisect_left`. (`bisect_right` đúng cho điều kiện `>`.)

### Lỗi 4: So sánh cả danh sách với một số

In [33]:
hours = [0, 1, 2, 3, 4]
try:
    print(sum(hours >= 2))         # sai: so sánh list với int
except TypeError as e:
    print("lỗi:", e)
print(sum(h >= 2 for h in hours))  # đúng: so sánh từng phần tử

lỗi: '>=' not supported between instances of 'list' and 'int'
3


**Vì sao sai:** `hours >= 2` so sánh một **danh sách** với một **số**, Python báo `TypeError`. Phải so sánh **từng phần tử** `h >= target` (qua vòng lặp hoặc biểu thức sinh).

### Lỗi 5: Sửa mảng đầu vào khi sắp xếp tại chỗ

In [34]:
hours = [5, 1, 4, 2, 2]
hours.sort()                       # sort() sửa trực tiếp hours (trả None)
print(hours)                       # [1, 2, 2, 4, 5]: thứ tự gốc bị mất
hours2 = [5, 1, 4, 2, 2]
s = sorted(hours2)                 # sorted() tạo danh sách mới
print(hours2, s)                   # hours2 giữ nguyên

[1, 2, 2, 4, 5]
[5, 1, 4, 2, 2] [1, 2, 2, 4, 5]


**Vì sao nên tránh:** `list.sort()` thay đổi dữ liệu đầu vào (có thể gây lỗi khi người gọi còn dùng thứ tự gốc, ví dụ cần chỉ số nhân viên). Dùng `sorted(...)` để giữ nguyên `hours`. Với bài này, thứ tự gốc không ảnh hưởng đáp án, nhưng đây là thói quen tốt.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm số phần tử thoả điều kiện" → `sum(điều_kiện for x in a)`. Đây là mẫu quen thuộc nhất của Python cho bài đếm.
- Đọc kỹ cụm **"ít nhất" (`>=`)**, **"nhiều hơn" (`>`)**, **"nhiều nhất" (`<=`)**, **"ít hơn" (`<`)**: chọn đúng toán tử.
- Nếu cần nhiều truy vấn "bao nhiêu phần tử `>= t`" trên cùng một mảng: **sắp xếp một lần + `bisect_left`** (hoặc mảng đếm + tổng hậu tố khi miền giá trị nhỏ).
- `bisect_left(a, x)` ⇒ vị trí đầu tiên `a[p] >= x`; `bisect_right(a, x)` ⇒ vị trí đầu tiên `a[p] > x`.
- Luôn thử các biên: `target = 0`, `target` bằng đúng một vài giá trị, `target` lớn hơn tất cả.

## 15. Biến thể của bài

1. **Đếm số nhân viên làm nhiều hơn hẳn `target` giờ** (`>`): đổi toán tử hoặc dùng `bisect_right`.
2. **Đếm số nhân viên làm trong khoảng `[a, b]` giờ:** `sum(a <= h <= b for h in hours)` hoặc `bisect_right(s, b) − bisect_left(s, a)`.
3. **Trả về danh sách chỉ số** các nhân viên đạt: `[i for i, h in enumerate(hours) if h >= target]`.
4. **Nhiều truy vấn `target`:** sắp xếp một lần rồi `bisect` cho từng truy vấn.
5. **Số giờ còn thiếu của mỗi nhân viên** để đạt `target`: `max(0, target − h)`; tổng thiếu `sum(max(0, target - h) for h in hours)`.
6. **Tìm `target` lớn nhất sao cho ít nhất `k` nhân viên đạt:** đó là phần tử lớn thứ `k` của `hours` (sắp xếp giảm dần lấy phần tử thứ `k − 1`).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng `for` | đếm bằng biến `count` | `O(n)` | `O(1)` | rõ ràng nhất |
| 2. `sum` boolean | `True` là `1` | `O(n)` | `O(1)` | **code nộp** |
| 3. Lọc rồi `len` | giữ danh sách đạt | `O(n)` | `O(n)` | cần chính danh sách |
| 4. Sắp xếp + `bisect_left` | các phần tử đạt nằm liền ở cuối | `O(n log n)` | `O(n)` | nhiều truy vấn |
| 5. Mảng đếm | `sum(cnt[target:])` | `O(n + V)` | `O(V)` | miền giá trị nhỏ, nhiều truy vấn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `sum([True, True, False])`, `3 >= 3`, `bisect_left([1, 2, 2, 4], 2)`, `bisect_right([1, 2, 2, 4], 2)`.

**Bài 2 (tính tay).** Với `hours = [3, 8, 5, 5, 0, 10]`, `target = 5`: đếm số người đạt; rồi đếm số người làm **nhiều hơn hẳn** 5 giờ.

**Bài 3 (điền TODO).** Hoàn thành `count_met_todo(hours, target)`.

In [35]:
def count_met_todo(hours, target):
    count = 0
    for h in hours:
        # TODO: nếu h đạt target (ít nhất target giờ) thì tăng count
        pass
    return count

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `count_met_bisect(hours, target)` bằng sắp xếp và tự cài `lower_bound` (không dùng thư viện `bisect`). Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_in_range(hours, a, b)`: số nhân viên có `a <= giờ <= b`. (b) Viết `answer_queries(hours, targets)` trả về danh sách đáp án cho **nhiều** `target` (sắp xếp một lần).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `2`; `True`; `1`; `3`. Bài 2: giờ `>= 5` là `8, 5, 5, 10` ⇒ `4`; giờ `> 5` là `8, 10` ⇒ `2`.
- **Mức 2 (vừa):** Bài 3: `if h >= target: count += 1`. Bài 4: copy `lower_bound` ở mục 3.6, rồi `len(s) - lower_bound(s, target)`.
- **Mức 3 (sâu):** Bài 5(a): `bisect_right(s, b) - bisect_left(s, a)` hoặc `sum(a <= h <= b for h in hours)`. Bài 5(b): `s = sorted(hours)` một lần, rồi `[len(s) - bisect_left(s, t) for t in targets]`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [36]:
from bisect import bisect_left, bisect_right

# Bài 1
assert sum([True, True, False]) == 2
assert (3 >= 3) is True
assert bisect_left([1, 2, 2, 4], 2) == 1
assert bisect_right([1, 2, 2, 4], 2) == 3

# Bài 2
h2 = [3, 8, 5, 5, 0, 10]
assert sum(h >= 5 for h in h2) == 4
assert sum(h > 5 for h in h2) == 2

# Bài 3
def count_met_todo_ans(hours, target):
    count = 0
    for h in hours:
        if h >= target:
            count += 1
    return count

assert count_met_todo_ans([0, 1, 2, 3, 4], 2) == 3
assert count_met_todo_ans([5, 1, 4, 2, 2], 6) == 0

# Bài 4
def lower_bound_ans(a, x):
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo

def count_met_bisect(hours, target):
    s = sorted(hours)
    return len(s) - lower_bound_ans(s, target)

assert count_met_bisect([0, 1, 2, 3, 4], 2) == 3
assert count_met_bisect([5, 1, 4, 2, 2], 6) == 0

# Bài 5(a)
def count_in_range(hours, a, b):
    s = sorted(hours)
    return bisect_right(s, b) - bisect_left(s, a)

assert count_in_range([3, 8, 5, 5, 0, 10], 5, 8) == 3
assert count_in_range([3, 8, 5, 5, 0, 10], 11, 20) == 0

# Bài 5(b)
def answer_queries(hours, targets):
    s = sorted(hours)
    return [len(s) - bisect_left(s, t) for t in targets]

assert answer_queries([3, 8, 5, 5, 0, 10], [0, 5, 6, 11]) == [6, 4, 2, 0]

random.seed(8)
for _ in range(300):
    hours, target = random.choice([gen_wide, gen_narrow, gen_tie])()
    assert count_met_bisect(hours, target) == Solution().numberOfEmployeesWhoMetTarget(hours, target)
    a, b = sorted([random.randint(0, 10), random.randint(0, 10)])
    assert count_in_range(hours, a, b) == sum(a <= h <= b for h in hours)
    ts = [random.randint(0, 100000) for _ in range(5)]
    assert answer_queries(hours, ts) == [Solution().numberOfEmployeesWhoMetTarget(hours, t) for t in ts]
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm số phần tử thoả điều kiện)

```python
from typing import List

class Solution:
    def numberOfEmployeesWhoMetTarget(self, hours: List[int], target: int) -> int:
        return sum(h >= target for h in hours)       # True == 1
```

**Checklist**

- [ ] "Ít nhất `target` giờ" dùng `>=` (không phải `>`)?
- [ ] Trả về một **số** (số nhân viên), không phải danh sách?
- [ ] So sánh **từng phần tử** `h >= target`, không so sánh cả danh sách với số?
- [ ] Nếu dùng `bisect`: `n − bisect_left(sorted_hours, target)` (không phải `bisect_right`)?
- [ ] Đã thử `target = 0`, `target` bằng đúng giờ của vài người, `target` lớn hơn tất cả?
- [ ] Không làm hỏng `hours` gốc (dùng `sorted`, không `sort`)?
- [ ] Signature đúng `numberOfEmployeesWhoMetTarget(self, hours: List[int], target: int) -> int` và có `from typing import List`?